# open-source-marginal-emissions

## weather_data_retrieval

The purpose of this notebook is to provide an interactive way to retrieve data from either:
* CDS API (Copernicus Data Store) for ERA5 data
* Open-Meteo API

### Code

#### Libraries

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import requests
import datetime
import time
import os
import sys
import cdsapi
import getpass
import calendar
import pathlib
import tempfile
import hashlib
import csv
from datetime import datetime, timedelta
from tqdm import tqdm
import speedtest
from concurrent.futures import ThreadPoolExecutor, as_completed


#### Paths

In [2]:
root_dir = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
data_dir = os.path.join(root_dir, "data")
raw_data_dir = os.path.join(data_dir, "raw")

print(raw_data_dir)

/Users/Daniel/Desktop/open-source-marginal-emissions/data/raw


#### Creds (temp notes)

url: https://cds.climate.copernicus.eu/api



key: 3dabbb79-833a-4098-bff3-a2c59d43389d

#### Implementation

#### MVP

##### Functions

In [3]:
# Authentication prompt and connection

def auth_prompt():
    """
    Prompt user for CDS API URL and key, and initialize cdsapi.Client.
    Parameters
    ----------
    None

    Returns
    -------
    client: cdsapi.Client
        Initialized CDS API client.

    """
    print("=" * 40)
    print("CDS API Authentication\n" + "=" * 40 + "\n")

    api_url = input("Enter your CDS API URL (default: https://cds.climate.copernicus.eu/api): ").strip()
    if not api_url:
        api_url = "https://cds.climate.copernicus.eu/api"

    api_key = getpass.getpass("Enter your CDS API key (the long token with hyphens): ").strip()
    client = cdsapi.Client(url=api_url, key=api_key)
    if client.progress and client.session is not None:
        print("CDS API Client initialized successfully.\n\n")
    return client

In [4]:
def format_coordinates(boundaries: list) -> str:
    """
    Extracts and formats coordinates as integers in N-W-S-E order

    Parameters
    ----------
    boundaries : list
        List of boundaries in the order [north, west, south, east]
    Returns
    -------
        str: Formatted string in the format 'N{north}W{west}S{south}E{east}'
    """
    north, west, south, east = boundaries
    return f"N{int(north)}W{int(west)}S{int(south)}E{int(east)}"

In [5]:
def format_duration(seconds: float) -> str:
    """
    Convert seconds to a nice Hh Mm Ss string (with decimal seconds).

    Parameters
    ----------
    seconds : float
        Duration in seconds.

    Returns
    -------
        str: Formatted duration string.
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60  # keep remainder as float
    if hours > 0:
        return f"{hours}h {minutes}m {secs:.1f}s"
    elif minutes > 0:
        return f"{minutes}m {secs:.1f}s"
    else:
        return f"{secs:.1f}s"

In [6]:
def create_download_hash(
        dataset: str,
        variables : list[str],
        boundaries : list[float]
    ) -> str:
    """
    Create a unique hash for the download parameters.

    Parameters
    ----------
    dataset : str
        The dataset name.
    variables : list[str]
        List of variable names.
    boundaries : list[float]
        List of boundaries [north, west, south, east].

    Returns
    -------
        str: A unique hash string representing the download parameters.
    """
    # Create unique string from all parameters
    param_string = f"{dataset}|{sorted(variables)}|{boundaries}"

    # Generate hash
    hash_object = hashlib.md5(param_string.encode())
    return hash_object.hexdigest()[:12]  # 12 characters

In [7]:
def estimate_grib_size_mb(
        variables_list: list[str],
        area_bounds_list: list[float]
    ) -> float:
    """
    Roughly estimate file size per month in MB

    Parameters
    ----------
    variables_list : list[str]
        List of variable names.
    area_bounds_list : list[float]
        List of boundaries [north, west, south, east].
    Returns
    -------
        float: Estimated file size in MB.
    """
    num_vars = len(variables_list)
    north, west, south, east = area_bounds_list

    lat_span = abs(north - south)
    lon_span = abs(east - west)

    area_fraction = (lat_span * lon_span) / (180 * 360)  # fraction of global

    base_size_per_var_global = 80  # MB per var per month (empirical)

    return num_vars * base_size_per_var_global * area_fraction


In [8]:
def estimate_era5_download(
        variables: list[str],
        area: list[float],
    ) -> tuple[float, float]:
    """
    Estimate GRIB size (MB) and total wall time (min) for CDS API requests.

    Parameters
    ----------
    variables : list[str]
        List of variable names.
    area : list[float]
        List of boundaries [north, west, south, east].

    Returns
    -------
        tuple: Estimated file size in MB and estimated time in minutes.
    """
    n_vars = len(variables)
    north, west, south, east = area
    area_fraction = (abs(north - south) * abs(east - west)) / (180 * 360)

    # Base size per variable per month (global, GRIB compressed)
    base_global_mb = 200  # empirical average for ERA5 single-levels

    # Estimate
    est_size_mb = base_global_mb * area_fraction * n_vars

    # Server prep + transfer time model
    # baseline 3 min overhead + 0.2 min/var + 0.03 min/MB (transfer)
    est_time_min = 3 + 0.2 * n_vars + 0.03 * est_size_mb

    return est_size_mb, est_time_min


In [9]:
def internet_speedtest(
        test_urls: list[str] = ["https://speedtest.london.linode.com/100MB-london.bin",
        "https://mirror.de.leaseweb.net/speedtest/100mb.bin",
        "https://ipv4.download.thinkbroadband.com/100MB.zip"],
        chunk_size: int = 10_000_000
        ) -> float:
    """
    Download ~100MB test file from a fast CDN to estimate speed (MB/s).

    Parameters
    ----------
    test_urls : list[str], optional
        List of URLs of the test files, by default ["https://speedtest.london.linode.com/100MB-london.bin"].
    chunk_size : int, optional
        Size of each chunk to download in bytes, by default 10,000,000.

    Returns
    -------
        float: Estimated download speed in Mbps.
    """
    print("="*40)
    print(f"Internet speed test\n" + "="*40)
    print("\n")
    for test_url in test_urls:
        try:
            print(f"Testing {test_url}\n\tMay take up to 15 seconds...")
            t0 = time.time()
            downloaded = 0
            with requests.get(test_url, stream=True, timeout=15, verify=True) as r:
                r.raise_for_status()
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        break
                downloaded += len(chunk)
                if downloaded >= 50_000_000:  # stop after ~50 MB
                    break
            elapsed = time.time() - t0
            mbps = (downloaded * 8 / 1e6) / elapsed
            print(f"\tRESULT: SUCCESS\nObserved Speed: {mbps:.1f} Mbps (based on {downloaded/1e6:.1f} MB in {elapsed:.1f}s)")
            return mbps
        except Exception as e:
            print(f"\tRESULT: FAILURE on {test_url}: {e}")
    print("All tests failed. Assuming 25 Mbps.")
    return 25.0


In [ ]:
def prepare_download(save_dir: str,
                     filename_base: str,
                     year: int,
                     month: int,
                     skip_all: bool = False,
                     overwrite_all: bool = False,
                     case_by_case: bool = False,
                     skipped_downloads: list = None
    ) -> tuple[bool, str]:
    """
    Check if a monthly ERA5 file already exists and decide whether to download.

    Parameters
    ----------
    save_dir : str
        Directory to save the downloaded file.
    filename_base : str
        Base name for the file.
    year : int
        Year of the data to download.
    month : int
        Month of the data to download.
    skip_all : bool, optional
        If True, skip all existing files without prompt, by default False.
    overwrite_all : bool, optional
        If True, overwrite all existing files without prompt, by default False.
    case_by_case : bool, optional
        If True, prompt user for each existing file, by default False.
    skipped_downloads : list, optional
        List to append skipped downloads, by default [].

    Returns
    -------
    tuple: (download: bool, save_path: str)
        download: Whether to perform the download.
        save_path: Full path for the target file.
    """
    if skipped_downloads is None:
        skipped_downloads = []

    save_path = os.path.join(save_dir, f"{filename_base}_{year}_{month}.grib")
    download = True

    # Handle existing file logic
    if os.path.exists(save_path):
        if skip_all:
            tqdm.write(f"Skipping existing file for {year}-{month}: {save_path}")
            skipped_downloads.append((year, month))
            download = False
            return download, save_path

        elif overwrite_all:
            tqdm.write(f"Overwriting existing file for {year}-{month}: {save_path}")
            return download, save_path

        elif case_by_case:
            while True:
                user_input = input(
                    f"\nFile already exists for {year}-{month}: {save_path}\n"
                    "Do you want to overwrite it? (y/n): "
                ).strip().lower()
                if user_input in ["y", "n"]:
                    break
                print("Invalid input. Please enter 'y' or 'n'.")
            if user_input == "n":
                tqdm.write(f"Skipping existing file for {year}-{month}: {save_path}")
                skipped_downloads.append((year, month))
                download = False
                return download, save_path
            else:
                tqdm.write(f"Overwriting existing file for {year}-{month}: {save_path}")
                return download, save_path

    return download, save_path


In [ ]:
def execute_download(session,
                     save_path: str,
                     dataset_name: str,
                     product_type: str,
                     variables: list[str],
                     days: list[str],
                     times: list[str],
                     grid_area: list[float],
                     year: int,
                     month: int,
                     data_download_format: str = "grib",
                     successful_downloads: list = None,
                     failed_downloads: list = None,
                     max_retries: int = 6,
                     retry_delay_sec: int = 15
    ) -> tuple[int, int, str]:
    """
    Execute a single ERA5 monthly download with retry logic.

    Parameters
    ----------
    session : cdsapi.Client
        Authenticated CDS API client.
    dataset_name : str
        Dataset name (e.g., 'reanalysis-era5-single-levels').
    product_type : str
        Product type (e.g., 'reanalysis').
    variables : list[str]
        List of variable names.
    days : list[str]
        List of days to download.
    times : list[str]
        List of times to download.
    grid_area : list[float]
        Geographic boundaries [north, west, south, east].
    data_download_format : str
        Format of the downloaded data (e.g., 'grib', 'netcdf').
    year : int
        Year of the data to download.
    month : int
        Month of the data to download.
    save_path : str
        Full path to save the downloaded file.
    successful_downloads : list, optional
        List to append successful downloads.
    failed_downloads : list, optional
        List to append failed downloads.
    max_retries : int, optional
        Maximum number of retry attempts, by default 6.
    retry_delay_sec : int, optional
        Delay in seconds between retries, by default 15.

    Returns
    -------
    (year, month, status): tuple
        status = "success" | "failed"
    """
    if successful_downloads is None:
        successful_downloads = []
    if failed_downloads is None:
        failed_downloads = []

    month_start = time.time()
    for attempt in range(1, max_retries + 1):
        try:
            tqdm.write(f"\tAttempt {attempt} of {max_retries} for {year}-{month}...")
            session.retrieve(
                dataset_name,
                {
                    "product_type": [product_type],
                    "variable": variables,
                    "year": str(year),
                    "month": [month],
                    "day": days,
                    "time": times,
                    "area": grid_area,
                    "format": data_download_format,
                },
                save_path,
            )
            elapsed = time.time() - month_start
            tqdm.write(f"SUCCESS: {year}-{month} in {format_duration(elapsed)}")
            successful_downloads.append((year, month))
            return (year, month, "success")

        except Exception as e:
            tqdm.write(f"WARNING: Attempt {attempt} failed for {year}-{month}: {e}")
            if attempt < max_retries:
                tqdm.write(f"\tWaiting {retry_delay_sec} seconds before retrying...")
                time.sleep(retry_delay_sec)
                try:
                    session = cdsapi.Client()
                    tqdm.write(f"\tRe-authenticated CDS API client.")
                except Exception as auth_e:
                    tqdm.write(f"\tWARNING: Re-authentication failed: {auth_e}")
            else:
                tqdm.write(f"FAILURE: all {max_retries} attempts failed for {year}-{month}.")
                failed_downloads.append((year, month))
                return (year, month, "failed")


In [ ]:
def download_month(session,
                   dataset_name: str,
                   product_type: str,
                   variables: list[str],
                   days: list[str],
                   times: list[str],
                   grid_area: list[float],
                   data_download_format: str,
                   save_dir: str,
                   filename_base: str,
                   year: int,
                   month: int,
                   skip_all: bool = False,
                   overwrite_all: bool = False,
                   case_by_case: bool = False,
                   skipped_downloads: list = None,
                   successful_downloads: list = None,
                   failed_downloads: list = None,
                   max_retries: int = 6,
                   retry_delay_sec: int = 10
    ) -> tuple[int, int, str]:
    """
    Orchestrate ERA5 monthly download: handle file checks, then execute download.

    Parameters
    ----------
    Combines parameters from `prepare_download` and `execute_download`.

    Returns
    -------
    (year, month, status): tuple
        status = "success" | "failed" | "skipped"

    """
    proceed, save_path = prepare_download(
        save_dir, filename_base, year, month,
        skip_all, overwrite_all, case_by_case, skipped_downloads
    )
    if not proceed:
        return (year, month, "skipped")

    return execute_download(
        session, dataset_name, product_type, variables, days, times,
        grid_area, data_download_format, year, month, save_path,
        successful_downloads, failed_downloads, max_retries, retry_delay_sec
    )

##### Implementation

In [10]:
# USER DEFINED INPUTS

variables = [
             'surface_pressure',
             # Cloud Vars
             'total_cloud_cover',
             'low_cloud_cover',
             'medium_cloud_cover',
             'high_cloud_cover',
            #  'fraction_of_cloud_cover' - likely not available for world data
            # Precipitation Vars
             'instantaneous_large_scale_surface_precipitation_fraction',
            #  'large_scale_precipitation', - likely not available for world data
             'convective_precipitation',
             'total_precipitation',
             'total_column_water',
             # Temperature Vars
             '2m_temperature',
            # Wind Vars
             '10m_u_component_of_wind',
             '10m_v_component_of_wind',
             '100m_u_component_of_wind',
             '100m_v_component_of_wind',
            # Radiation Vars
             'clear_sky_direct_solar_radiation_at_surface',

             'surface_net_solar_radiation',
             'surface_net_solar_radiation_clear_sky',
             'surface_net_thermal_radiation',
             'surface_net_thermal_radiation_clear_sky',

             'surface_solar_radiation_downwards',
             'surface_solar_radiation_downward_clear_sky',

             'surface_thermal_radiation_downwards',
             'surface_thermal_radiation_downward_clear_sky',

             'top_net_solar_radiation',
             'top_net_solar_radiation_clear_sky',

             'top_net_thermal_radiation',
             'top_net_thermal_radiation_clear_sky',

             'toa_incident_solar_radiation',
             'total_sky_direct_solar_radiation_at_surface',
]

In [ ]:
# USER DEFINED INPUTS
start_date = '2025-01-01'
end_date = '2025-06-01'
# datetime.date.today().strftime('%Y-%m-%d')

north_latitude_boundary = 38.0
south_latitude_boundary = 6.0
east_longitude_boundary = 98.0
west_longitude_boundary = 68.0

save_dir = raw_data_dir

In [12]:
# HARD CODED INPUTS
era5_world_dataset  = 'reanalysis-era5-single-levels'
era5_world_grid_resolution = [0.25, 0.25]   # lat, lon
era5_land_dataset = 'reanalysis-era5-land'
era5_land_grid_resolution = [0.1, 0.1]     # lat, lon
data_download_format = "grib"      # 'netcdf' still experimental
grid_area = [north_latitude_boundary, west_longitude_boundary,
        south_latitude_boundary, east_longitude_boundary]

days = [f"{d:02d}" for d in range(1, 32)]
times = [f"{h:02d}:00" for h in range(24)]

In [ ]:
start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
years_months = []

upper_date_boundary = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=8)

if end_dt > upper_date_boundary:
    print(f"End date {end_dt.strftime('%Y-%m-%d')} is beyond the upper data availability boundary of {upper_date_boundary.strftime('%Y-%m-%d')}. Adjusting end date.")
    end_dt = upper_date_boundary

current = start_dt.replace(day=1)
while current <= end_dt:
    years_months.append((current.year, f"{current.month:02d}"))
    next_month = current.month + 1
    next_year = current.year + (1 if next_month > 12 else 0)
    current = current.replace(year=next_year, month=(next_month - 12 if next_month > 12 else next_month))


In [14]:
# USER DEFINED INPUTS
dataset = era5_world_dataset

In [15]:
# Example usage:
coordinates = format_coordinates(grid_area)  # Returns: "38-68-6-98"
dataset_short = 'era5_world'
download_hash = create_download_hash(
    dataset=dataset,
    variables=variables,
    boundaries=grid_area,
)

filename_base = f"{dataset_short}_{coordinates}_{download_hash}"
print("Filename base:", filename_base)

Filename base: era5_world_N38W68S6E98_f6c9e6dda20a


In [16]:
down_speed_mbps = internet_speedtest()

Internet speed test


Testing https://speedtest.london.linode.com/100MB-london.bin
	May take up to 15 seconds...
	RESULT: FAILURE on https://speedtest.london.linode.com/100MB-london.bin: 404 Client Error: Not Found for url: https://speedtest.london.linode.com/100MB-london.bin
Testing https://mirror.de.leaseweb.net/speedtest/100mb.bin
	May take up to 15 seconds...
	RESULT: FAILURE on https://mirror.de.leaseweb.net/speedtest/100mb.bin: 404 Client Error: Not Found for url: https://mirror.de.leaseweb.net/speedtest/100mb.bin
Testing https://ipv4.download.thinkbroadband.com/100MB.zip
	May take up to 15 seconds...
	RESULT: FAILURE on https://ipv4.download.thinkbroadband.com/100MB.zip: HTTPSConnectionPool(host='ipv4.download.thinkbroadband.com', port=443): Max retries exceeded with url: /100MB.zip (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'ipv4.download.thinkbroadband.com'. (_ssl.

In [17]:
est_size_mb, est_time_min = estimate_era5_download(variables, grid_area)
n_files = len(years_months)
est_total_size_mb = est_size_mb * n_files
est_total_time_min = est_time_min * n_files

In [18]:
print("\nEstimated output summary:")
print(f"   Number of months/files : {n_files}")
print(f"   Estimated size per file : {est_size_mb:.1f} MB")
print(f"   Estimated total size     : {est_total_size_mb:.1f} MB")
print(f"   Estimated time per file  : {est_time_min:.1f} min")
print(f"   Estimated total time     : {format_duration(est_total_time_min * 60)}\n")


Estimated output summary:
   Number of months/files : 90
   Estimated size per file : 85.9 MB
   Estimated total size     : 7733.3 MB
   Estimated time per file  : 11.4 min
   Estimated total time     : 17h 4m 0.0s



In [19]:
max_retries = 6
retry_delay_sec = 10

In [ ]:
print("=" *40 + "\nChecking for existing files..." + "\n" + "=" *40)

existing_files = [
    f for f in os.listdir(save_dir)
    if f.startswith(filename_base) and f.endswith(".grib")
]
if existing_files:
    print(f"\nFound {len(existing_files)} existing ERA5 files for this configuration.")
    print("Choose how to proceed:")
    print("\t1. Overwrite all existing files")
    print("\t2. Skip all existing files")
    print("\t3. Case-by-case confirmation")

    while True:
        print("")
        choice = input("Enter choice (1/2/3): ").strip()
        if choice in ["1", "2", "3"]:
            break
        print("Invalid input. Please enter 1, 2, or 3.")

    overwrite_all = (choice == "1")
    skip_all = (choice == "2")
    case_by_case = (choice == "3")
else:
    print("No existing files found. Proceeding with downloads.")
    overwrite_all = skip_all = case_by_case = False


# === MAIN DOWNLOAD LOOP WITH PROGRESS BAR ===
session = auth_prompt()

overall_start_time = time.time()
failed_downloads = []
successful_downloads = []
skipped_downloads = []

max_workers = 2  # Keep small to avoid CDS throttling
futures = []

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = []
with tqdm(total=n_files, desc="Monthly ERA5 data requests progress", unit="file") as pbar:

    for year, month in years_months:
        save_path = os.path.join(save_dir, f"{filename_base}_{year}_{month}.grib")
        if os.path.exists(save_path):
            if overwrite_all:
                tqdm.write(f"\nOverwriting existing file for {year}-{month}: {save_path}\n")
            elif skip_all:
                tqdm.write(f"\nSkipping existing file for {year}-{month}: {save_path}\n")
                skipped_downloads.append((year, month))
                pbar.update(1)
                continue
            elif case_by_case:
                while True:
                    user_input = input(f"\nFile already exists for {year}-{month}: {save_path}\nDo you want to overwrite it? (y/n): ").strip().lower()
                    if user_input in ['y', 'n']:
                        break
                    print("Invalid input. Please enter 'y' or 'n'.")
                if user_input == 'n':
                    tqdm.write(f"Skipping existing file for {year}-{month}: {save_path}\n")
                    pbar.update(1)
                    continue
                else:
                    skipped_downloads.append((year, month))
                    tqdm.write(f"Overwriting existing file for {year}-{month}: {save_path}\n")

        tqdm.write(f"\nProcessing {year}-{month} ...\n")
        month_start = time.time()

        success = False
        for attempt in range(1, max_retries + 1):
            try:
                tqdm.write(f"\tAttempt {attempt} of {max_retries} for {year}-{month}...")
                session.retrieve(
                    era5_world_dataset,
                    {
                        "product_type": ["reanalysis"],
                        "variable": variables,
                        "year": str(year),
                        "month": [month],
                        "day": days,
                        "time": times,
                        "area": grid_area,
                        "format": data_download_format,
                    },
                    save_path,
                )
                elapsed = time.time() - month_start
                tqdm.write(f"SUCCESS: Completed {year}-{month} in {format_duration(elapsed)}")
                success = True
                successful_downloads.append((year, month))
                break
            except Exception as e:
                tqdm.write(f"\tWARNING: Attempt {attempt} failed for {year}-{month}: {e}")
                if attempt < max_retries:
                    tqdm.write(f"\tWaiting {retry_delay_sec} seconds before retrying...")
                    time.sleep(retry_delay_sec)
                    try:
                        session = cdsapi.Client()
                        tqdm.write(f"\tSUCCESS: Re-authenticated CDS API client.")
                    except Exception as auth_e:
                        tqdm.write(f"\tWARNING: Re-authentication failed: {auth_e}")
                else:
                    tqdm.write(f"FAILURE - All {max_retries} attempts failed for {year}-{month}: {e}")
                    failed_downloads.append((year, month))

        pbar.update(1)

overall_elapsed = time.time() - overall_start_time

Checking for existing files...
No existing files found. Proceeding with downloads.
CDS API Authentication

CDS API Client initialized successfully.




Monthly ERA5 data requests progress:   0%|          | 0/90 [00:00<?, ?file/s]


Processing 2018-01 ...

	Attempt 1 of 6 for 2018-01...


2025-10-22 17:30:54,277 INFO Request ID is b523ebed-4728-4c9c-be17-d3ef729bf9ab
2025-10-22 17:30:54,767 INFO status has been updated to accepted
2025-10-22 17:31:03,615 INFO status has been updated to running
2025-10-22 17:49:37,105 INFO status has been updated to successful


7c04ecbaa52c9c244a743a4640016322.grib:   0%|          | 0.00/645M [00:00<?, ?B/s]

Monthly ERA5 data requests progress:   1%|          | 1/90 [18:55<28:04:47, 1135.82s/file]

SUCCESS: Completed 2018-01 in 18m 55.8s

Processing 2018-02 ...

	Attempt 1 of 6 for 2018-02...


2025-10-22 17:49:50,267 INFO Request ID is fc2ed7c6-f984-4418-8949-643e74999a8c
2025-10-22 17:49:50,392 INFO status has been updated to accepted
2025-10-22 17:50:04,123 INFO status has been updated to running
Monthly ERA5 data requests progress:   1%|          | 1/90 [19:42<29:13:20, 1182.03s/file]


KeyboardInterrupt: 

In [ ]:
#=== SUMMARY ===
print("\n📊 Download Summary")
print(f"   Total months requested : {n_files}")
print(f"   Successful downloads   : {len(successful_downloads)}")
print(f"   Skipped downloads      : {len(skipped_downloads)}")
print(f"   Failed downloads       : {len(failed_downloads)}")
print(f"   Total time elapsed     : {format_duration(overall_elapsed)}")

if failed_downloads:
    print("   Failed months:", ", ".join([f"{y}-{m}" for y, m in failed_downloads]))

print(f"Files Downloaded to {save_dir}:")
for year, month in successful_downloads:
    print(f"   - {filename_base}_{year}_{month}.grib")


📊 Download Summary
   Total months requested : 3
   Successful downloads   : 1
   Skipped downloads      : 0
   Failed downloads       : 0
   Total time elapsed     : 25.7s
Files Downloaded to /Users/Daniel/Desktop/open-source-marginal-emissions/data/raw:
   - era5_world_N38W68S6E98_010b38615413_2025_01.grib


#### Variables

##### Define Variables

In [42]:
user_choice = [{
    'weather_data_provider': None,      # e.g. 'Open-Meteo' or 'ERA5'
    'weather_dataset': None,           # e.g. 'era5-land or era5-world
    'temporal_resolution': None,        # '2H, '1H', '30min'
    'spatial_resolution': None,         # '0.25deg', '0.1deg'
    'variables': [None],                # e.g. ['temperature_2m', 'precipitation']
    'date_range': {
        'start_date': None,
        'end_date': None
    },
    'dataset_temporal_split': None,        # how many individual time segments to split the dataset into
    'geographical_area': {
        'latitude_n': None,
        'latitude_s': None,
        'longitude_e': None,
        'longitude_w': None
    }
}]

In [43]:
weather_data_providers = ['Open-Meteo', 'ERA5']
weather_data_providers_labels = {
    'Open-Meteo': '1. Open-Meteo',
    'ERA5': '2. ERA5'
}
weather_data_provider_aliases = {
    'Open-Meteo': {'open_meteo', '1', 'openmeteo', 'om', 'open', 'open-meteo'},
    'ERA5': {'era5', '2', 'era5land', 'era5world', 'era5', 'e5', 'era-5', 'era_5'}
}

In [44]:
era5_weather_datasets = ['ERA5-Land', 'ERA5-World', 'ERA5-Land-then-World', 'ERA5-World-then-Land']
era5_weather_datasets_labels = {
    'ERA5-Land': '1. ERA5-Land',
    'ERA5-World': '2. ERA5-World',
    'ERA5-Land-then-World': '3. ERA5-Land-then-World',
    'ERA5-World-then-Land': '4. ERA5-World-then-Land'
}
era5_weather_dataset_aliases = {
    'ERA5-Land': {'era5land', 'land', '1', 'era5-land', 'era5_land'},
    'ERA5-World': {'era5world', 'world', '2', 'era5-world', 'era5_world'},
    'ERA5-Land-then-World': {'era5landthenworld', 'landthenworld', '3', 'era5-land-then-world', 'era5_land_then_world'},
    'ERA5-World-then-Land': {'era5worldthenland', 'worldthenland', '4', 'era5-world-then-land', 'era5_world_then_land'}
}

In [45]:
temporal_variables = [120, 60, 30] # in minutes
temporal_variable_labels = {
    120: '2-hourly (2h)',
    60: 'hourly (1h)',
    30: 'half-hourly (30m)'
}
temporal_variable_aliases = {
    120: {'120', '2h', '2hr', '2hour', '2hours', '2-hourly', 'two-hourly', 'twohourly', 'two_hourly'},
    60: {'60', 'hourly', '1h', '1hr', '1hour', '1hours', 'one-hourly', 'onehourly', 'one_hourly'},
    30: {'30', 'half-hourly', 'halfhourly', 'half_hourly', '30m', '30min', '30minute', '30minutes'}
}

#### Functions

##### Weather Dataset Specification

In [46]:
def weather_dataset_specification_intro():
    print("\n" + ("=" * 50))
    print("Weather Data Retrieval")
    print("=" * 50 + "\n")
    print("Welcome to the Weather Data Retrieval Tool!")
    print("This tool allows you to retrieve historical weather data from various providers.")
    pass

In [47]:
def weather_dataset_specification_providers_choice():
    print("Please choose a weather data provider from one of the following options:")
    print(weather_data_providers_labels.values())
    choice = input("Enter your choice: ")
    failed_attempts = 0
    while choice not in weather_data_provider_aliases:
        print("Invalid choice. Please enter the exact name or number of the weather data provider.")
        failed_attempts += 1
        print(f"Failed attempts: {failed_attempts}/5")
        if failed_attempts >= 5:
            print("Too many failed attempts. Exiting the program.")
            exit()
        choice = input("\nEnter your choice: ")
    return choice

In [48]:
def era5_weather_dataset_choice():
    print("Please choose an ERA5 weather dataset from one of the following options:")
    print(era5_weather_datasets_labels.values())
    failed_attempts = 0
    choice = input("Enter your choice: ")
    while choice not in era5_weather_dataset_aliases:
        print("Invalid choice. Please enter the exact name or number of the ERA5 weather dataset.")
        failed_attempts += 1
        print(f"Failed attempts: {failed_attempts}/5")
        if failed_attempts >= 5:
            print("Too many failed attempts. Exiting the program.")
            exit()
        choice = input("\nEnter your choice: ")
    return choice

In [ ]:
def weather_data_specification():
    weather_dataset_specification_intro()
    weather_data_provider = None
    weather_dataset = None
    end_loop = False
    while end_loop == False:
        weather_data_provider = weather_dataset_specification_providers_choice()
        if weather_data_provider == 'Open-Meteo':
            print("Open-Meteo data retrieval is not implemented yet. - Please choose another provider.")
            raise NotImplementedError("Open-Meteo data retrieval is not implemented yet.")
            # Implement Open-Meteo data retrieval here
        elif weather_data_provider == 'ERA5':
            weather_dataset = era5_weather_dataset_choice()
            end_loop = True
        else:
            raise ValueError("All previous input validation failed - recheck logic.")
            end_loop = True
    return weather_data_provider, weather_dataset

: 

In [ ]:
weather_data_specification()


Weather Data Retrieval

Welcome to the Weather Data Retrieval Tool!
This tool allows you to retrieve historical weather data from various providers.
Please choose a weather data provider from one of the following options:
dict_values(['1. Open-Meteo', '2. ERA5'])
Invalid choice. Please enter the exact name or number of the weather data provider.
Failed attempts: 1/5
Invalid choice. Please enter the exact name or number of the weather data provider.
Failed attempts: 2/5
Invalid choice. Please enter the exact name or number of the weather data provider.
Failed attempts: 3/5
Invalid choice. Please enter the exact name or number of the weather data provider.
Failed attempts: 4/5
Invalid choice. Please enter the exact name or number of the weather data provider.
Failed attempts: 5/5
Too many failed attempts. Exiting the program.
Invalid choice. Please enter the exact name or number of the weather data provider.
Failed attempts: 6/5
Too many failed attempts. Exiting the program.
Invalid cho

What data source would you like to use?
* Open-Meteo
* CDS API (ERA5 datasets)
    * IF USING CDS API:
    * What dataset would you like to use?
        * ERA5-Land only
        * ERA5-World only
        * ERA5-Land where available, otherwise ERA5-World
        * ERA5-World where available, otherwise ERA5-Land

Options:
open-meteo
era5-land
era5-world
era5-land-then-era5-world
era5-world-then-era5-land

Time Period:
Over what time period would you like to retrieve data?
* Start date (YYYY-MM-DD):
* End date (YYYY-MM-DD):

How would you like to  specify the goegraphical area over which to retrieve data?
* Provide bounding box coordinates
* Select bounding box on map
* Provide Country Name
* Provide City and Country Name 

Over what geographical area would you like to retrieve data?
* North latitude (degrees):
* South latitude (degrees):
* East longitude (degrees):
* West longitude (degrees):

* What temporal resolution would you like the data at?
    * 2-Hourly (aggregates)
    * Hourly (native)
    * Half-hourly (interpolated)

* What geographic resolution would you like the data at?
    * 0.1° x 0.1° (approx. 11km x 11km) (only native for ERA5-Land)
    * 0.25° x 0.25° (approx. 28km x 28km)
    * 0.5° x 0.5° (approx. 55km x 55km)
    * 1.0° x 1.0° (approx. 111km x 111km)

| NAME | VAR NAME | Units
|------|----------|------|
| Surface pressure | surface_pressure | Pa
| Total cloud cover | total_cloud_cover | 0-1
| 10 metre U wind component | 10m_u_component_of_wind | m/s
| 10 metre V wind component | 10m_v_component_of_wind | m/s
| 2 metre temperature | 2m_temperature | K
| Low cloud cover | low_cloud_cover | 0-1
| Medium cloud cover | medium_cloud_cover | 0-1
| High cloud cover | high_cloud_cover | 0-1
| Instantaneous large-scale surface precipitation fraction | instantaneous_large_scale_surface_precipitation_fraction | 0-1
| 100 metre U wind component | 100m_u_component_of_wind | m/s
| 100 metre V wind component | 100m_v_component_of_wind | m/s
| Surface solar radiation downwards | surface_solar_radiation_downwards | J m**-2
| Surface thermal radiation downwards | surface_thermal_radiation_downwards | J m**-2
| Surface net solar radiation | surface_net_solar_radiation | J m**-2
| Top net solar radiation | top_net_solar_radiation | J m**-2
| Top net thermal radiation | top_net_thermal_radiation | J m**-2
| Top net solar radiation, clear sky | top_net_solar_radiation_clear_sky | J m**-2
| Top net thermal radiation, clear sky | top_net_thermal_radiation_clear_sky | J m**-2
| Surface net solar radiation, clear sky | surface_net_solar_radiation_clear_sky | J m**-2
| Surface net thermal radiation, clear sky | surface_net_thermal_radiation_clear_sky | J m**-2
| TOA incident solar radiation | toa_incident_solar_radiation | J m**-2
| Total sky direct solar radiation at surface | total_sky_direct_solar_radiation_at_surface | J m**-2
| Clear-sky direct solar radiation at surface | clear_sky_direct_solar_radiation_at_surface | J m**-2
| Surface solar radiation downward clear-sky  | surface_solar_radiation_downward_clear_sky	 | J m**-2
| Surface thermal radiation downward clear-sky | surface_thermal_radiation_downward_clear_sky | J m**-2
| Large-scale precipitation | large_scale_precipitation | kg m**-2
| Convective precipitation | convective_precipitation | kg m**-2
| Total precipitation | total_precipitation | kg m**-2
| Total column water | total_column_water | kg m**-2
| Fraction of cloud cover | fraction_of_cloud_cover | 0-1